In [ ]:
import json
from typing import TypedDict, List, Literal, Annotated, Sequence, Optional, Dict, Any
from uuid import uuid4
from langchain_core.messages import BaseMessage, SystemMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.types import Command
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
checkpointer = MemorySaver()

In [ ]:
load_dotenv()
MAX_DISCUSSION_TURNS = 6
MAX_CODING_TURNS = 6
MAX_REVIEW_TURNS = 6

MIN_DISCUSSION_TURNS = 2
MIN_CODING_TURNS = 2
MIN_REVIEW_TURNS = 2

In [ ]:
BASE_PROMPT = """
You are a senior technical interviewer.
Ask concise, rigorous questions aligned to the current interview phase.
Do not provide complete solutions for the candidate.
"""

In [ ]:
# Legacy tool-based phase controls have been retired.
# Phase transitions are now controlled by structured assessments plus deterministic validators.

In [ ]:
class InterviewAgentState(TypedDict):
    session_id: str
    user_id: str

    phase: Literal["PROBLEM_DISCUSSION", "CODING", "REVIEW", "FEEDBACK"]
    messages: Annotated[Sequence[BaseMessage], add_messages]

    problem_statement: str
    problem_references: Optional[Dict[str, Any]]

    user_code: Optional[str]
    user_code_output: Optional[Dict[str, Any]]
    execution_attempts: int

    discussion_turns: int
    review_turns: int
    coding_turns: int

    discussion_assessment: Optional[Dict[str, Any]]
    coding_assessment: Optional[Dict[str, Any]]
    review_assessment: Optional[Dict[str, Any]]
    transition_reason: Optional[str]
    unmet_criteria: Optional[List[str]]

    internal_evaluation: Optional[Dict[str, Any]]
    feedback: Optional[Dict[str, Any]]

    difficulty: Optional[str]
    expected_time_minutes: Optional[int]
    total_time_spent_sec: Optional[int]
    total_submissions: Optional[int]
    hints_used: Optional[int]

    interview_start_time: float
    timer_expired: bool

    ready_for_coding: bool
    ready_for_review: bool
    ready_for_feedback: bool

In [ ]:
class ScoreWithNotes(BaseModel):
    """A score accompanied by evaluator notes."""
    score: int = Field(ge=0, le=10, description="Integer score from 0 to 10")
    notes: str = Field(min_length=1, description="Brief evaluator notes justifying this score")


class ComplexityScore(BaseModel):
    """Complexity analysis score with identified complexities."""
    score: int = Field(ge=0, le=10, description="Integer score from 0 to 10 evaluating complexity understanding")
    time_complexity: str = Field(min_length=1, description="The time complexity, e.g. O(n), O(n log n)")
    space_complexity: str = Field(min_length=1, description="The space complexity, e.g. O(1), O(n)")
    notes: str = Field(min_length=1, description="Notes on complexity analysis")


class Scores(BaseModel):
    """All evaluation scores."""
    problem_solving: ScoreWithNotes
    complexity_analysis: ComplexityScore
    communication: ScoreWithNotes


class StrengthItem(BaseModel):
    """A specific strength demonstrated by the candidate."""
    category: str = Field(min_length=1)
    title: str = Field(min_length=1)
    description: str = Field(min_length=1)
    impact: Literal["high", "medium", "low"]


class WeaknessItem(BaseModel):
    """A specific area where the candidate needs improvement."""
    category: str = Field(min_length=1)
    title: str = Field(min_length=1)
    description: str = Field(min_length=1)
    severity: Literal["high", "medium", "low"]


class ComplexityMetric(BaseModel):
    """Runtime or memory complexity assessment."""
    value: str = Field(min_length=1)
    status: Literal["optimal", "acceptable", "suboptimal"]


class KeyMetrics(BaseModel):
    """Key performance metrics for the candidate's solution."""
    runtime_complexity: ComplexityMetric
    memory_efficiency: ComplexityMetric
    coding_speed_percentile: int = Field(ge=0, le=100)


class Verdict(BaseModel):
    """Final hiring verdict."""
    decision: Literal["Strong Hire", "Hire", "Lean Hire", "Lean No Hire", "No Hire", "Strong No Hire"]
    confidence: float = Field(ge=0.0, le=1.0)
    summary: str = Field(min_length=1)


class SessionSummary(BaseModel):
    """High-level summary of the interview session."""
    overall_score: int = Field(ge=0, le=100)
    performance_label: Literal["Exceptional", "Strong Performance", "Adequate", "Below Expectations", "Poor"]
    difficulty: Literal["Easy", "Medium", "Hard"]
    time_spent_seconds: int = Field(ge=0)


class FeedbackItem(BaseModel):
    """Comprehensive structured feedback."""
    session_summary: SessionSummary
    scores: Scores
    strengths: List[StrengthItem]
    weaknesses: List[WeaknessItem]
    key_metrics: KeyMetrics
    final_verdict: Verdict


class FeedbackResponseFormat(BaseModel):
    """Response schema for FEEDBACK phase."""
    response: str = Field(min_length=1, description="Brief conversational closing summary.")
    feedback: FeedbackItem

In [ ]:
class InternalEvaluationFormat(BaseModel):
    """Structured output schema for internal evaluation node."""
    problem_solving: ScoreWithNotes
    complexity_analysis: ComplexityScore
    communication: ScoreWithNotes
    strengths: List[StrengthItem]
    weaknesses: List[WeaknessItem]
    runtime_complexity: ComplexityMetric
    memory_efficiency: ComplexityMetric


class CriterionAssessment(BaseModel):
    completed: bool = Field(description="Whether this criterion is satisfied")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence for this criterion")
    reason: str = Field(min_length=1, description="Short evidence-backed rationale")


class DiscussionAssessment(BaseModel):
    approach_explained: CriterionAssessment
    edge_cases_discussed: CriterionAssessment
    complexity_discussed: CriterionAssessment


class CodingAssessment(BaseModel):
    code_submitted: CriterionAssessment
    walkthrough_provided: CriterionAssessment
    correctness_discussed: CriterionAssessment


class ReviewAssessment(BaseModel):
    optimization_discussed: CriterionAssessment
    edge_case_validation: CriterionAssessment
    final_complexity_summary: CriterionAssessment

In [ ]:
# No transition tools are required in the hybrid controller path.

In [ ]:
# base_llm = ChatGroq(
#     model="llama-3.1-8b-instant"
# )
base_llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-lite-latest",
    # model = "gemini-3.1-flash-lite-preview"
 )

In [ ]:
def _format_problem_context(state: InterviewAgentState) -> str:
    references = state.get("problem_references") or {}
    title = references.get("title") if isinstance(references, dict) else None
    title = title or "Unknown Problem"
    refs_pretty = json.dumps(references, indent=2, default=str)
    return f"Problem Title: {title}\\n\\nProblem References:\\n{refs_pretty}"

def _looks_like_code_submission(user_text: str) -> bool:
    stripped = user_text.strip()
    code_markers = ("def ", "class ", "for ", "while ", "if ", "return ", "public class", "#include", "function ")
    if "\\n" in stripped and any(marker in stripped for marker in code_markers):
        return True
    if stripped.startswith(("def ", "class ", "public class", "#include", "function ")):
        return True
    return False

def _validate_criteria(assessment: Dict[str, Any], min_confidence: float = 0.65) -> tuple[bool, List[str]]:
    unmet: List[str] = []
    for criterion, payload in assessment.items():
        if not isinstance(payload, dict):
            unmet.append(criterion)
            continue
        completed = bool(payload.get("completed"))
        confidence = float(payload.get("confidence", 0.0))
        reason = str(payload.get("reason", "")).strip()
        if (not completed) or (confidence < min_confidence) or (not reason):
            unmet.append(criterion)
    return len(unmet) == 0, unmet

def _build_assessment_prompt(phase: str, requirements: List[str], user_text: str, state: InterviewAgentState) -> str:
    problem_statement = state.get("problem_statement") or ""
    context = _format_problem_context(state)
    req_lines = "\\n".join(f"- {item}" for item in requirements)
    return f"""
You are an assessment engine.
Phase: {phase}

Assess whether the candidate response satisfies each requirement.
Return only structured output according to the provided schema.
Be strict and avoid optimistic scoring.

Requirements:
{req_lines}

Problem Statement:
{problem_statement}

{context}

Candidate Message:
{user_text}
"""

def _assess_discussion_state(user_text: str, state: InterviewAgentState) -> Dict[str, Any]:
    prompt = _build_assessment_prompt(
        phase="PROBLEM_DISCUSSION",
        requirements=[
            "Candidate explained a valid approach.",
            "Candidate discussed meaningful edge cases.",
            "Candidate discussed both time and space complexity.",
        ],
        user_text=user_text,
        state=state,
    )
    result = base_llm.with_structured_output(DiscussionAssessment).invoke([
        SystemMessage(content=prompt),
        HumanMessage(content="Assess discussion criteria."),
    ])
    payload = result.model_dump() if hasattr(result, "model_dump") else result
    return payload if isinstance(payload, dict) else {}

def _assess_coding_state(user_text: str, state: InterviewAgentState) -> Dict[str, Any]:
    prompt = _build_assessment_prompt(
        phase="CODING",
        requirements=[
            "Candidate submitted code or concrete implementation details.",
            "Candidate provided a walkthrough of implementation flow.",
            "Candidate discussed correctness and test reasoning.",
        ],
        user_text=user_text,
        state=state,
    )
    result = base_llm.with_structured_output(CodingAssessment).invoke([
        SystemMessage(content=prompt),
        HumanMessage(content="Assess coding criteria."),
    ])
    payload = result.model_dump() if hasattr(result, "model_dump") else result
    return payload if isinstance(payload, dict) else {}

def _assess_review_state(user_text: str, state: InterviewAgentState) -> Dict[str, Any]:
    prompt = _build_assessment_prompt(
        phase="REVIEW",
        requirements=[
            "Candidate discussed optimization opportunities and trade-offs.",
            "Candidate validated important edge cases.",
            "Candidate summarized final time and space complexity.",
        ],
        user_text=user_text,
        state=state,
    )
    result = base_llm.with_structured_output(ReviewAssessment).invoke([
        SystemMessage(content=prompt),
        HumanMessage(content="Assess review criteria."),
    ])
    payload = result.model_dump() if hasattr(result, "model_dump") else result
    return payload if isinstance(payload, dict) else {}

In [ ]:
def interview_init_node(state: InterviewAgentState) -> dict:
    existing_messages = list(state.get("messages", []))
    if existing_messages:
        return {"phase": "PROBLEM_DISCUSSION"}

    init_prompt = """
    CURRENT PHASE: INTERVIEW_INITIALIZATION

    Your responsibilities:
    1. Briefly greet the candidate.
    2. Explain that this will be a structured technical interview.
    3. Mention that you will first discuss the approach before writing code.
    4. Clearly present the problem statement.
    5. Ask the candidate to:
    - Restate the problem in their own words
    OR
    - Propose an initial approach.
    Do NOT:
    - Give hints.
    - Suggest the optimal approach.
    - Ask for full code.
    - Evaluate the candidate.
    End your message with a clear open-ended question.
    """

    problem_statement = state.get("problem_statement") or ""
    problem_context = _format_problem_context(state)

    final_prompt = (
        f"{BASE_PROMPT}\n{init_prompt}\n"
        f"{problem_context}\n\n"
        f"Problem Statement:\n{problem_statement}"
    )

    messages = [
        SystemMessage(content=final_prompt),
        HumanMessage(content="Please start the interview."),
    ]

    res = base_llm.invoke(messages)

    print(f"AI message: {res.content}")

    return {
        "phase":"PROBLEM_DISCUSSION",
        "messages":[res]
    }

In [ ]:
def problem_discussion_node(state: InterviewAgentState) -> dict:
    phase_discussion_prompt = """
    CURRENT PHASE: PROBLEM_DISCUSSION

    Your responsibilities:
    - Ask the candidate about their approach.
    - Clarify assumptions.
    - Identify potential edge cases.
    - Ask for time and space complexity.
    - Challenge vague reasoning.
    """

    problem_statement = state.get("problem_statement") or ""
    problem_context = _format_problem_context(state)

    final_prompt = (
        f"{BASE_PROMPT}\\n{phase_discussion_prompt}\\n"
        f"{problem_context}\\n\\n"
        f"Problem Statement:\\n{problem_statement}"
    )

    prior_messages = list(state.get("messages", []))
    current_turns = state.get("discussion_turns", 0)
    if not prior_messages or not isinstance(prior_messages[-1], HumanMessage):
        return {
            "phase": "PROBLEM_DISCUSSION",
            "discussion_turns": current_turns,
            "transition_reason": "awaiting_user_message",
        }

    latest_user_content = prior_messages[-1].content
    latest_user_text = latest_user_content if isinstance(latest_user_content, str) else str(latest_user_content)
    new_turns = current_turns + 1

    if _looks_like_code_submission(latest_user_text):
        return {
            "phase": "CODING",
            "ready_for_coding": True,
            "discussion_turns": new_turns,
            "transition_reason": "code_submission_detected",
            "messages": [
                AIMessage(
                    content="Thanks for sharing code. We are now in the coding phase. Please walk through your implementation and reasoning."
                )
            ],
        }

    assessment = _assess_discussion_state(latest_user_text, state)
    validated, unmet = _validate_criteria(assessment)
    min_turn_gate = new_turns >= MIN_DISCUSSION_TURNS

    if validated and min_turn_gate:
        return {
            "messages": [
                AIMessage(
                    content="Great discussion. Let's move to the coding phase. Please implement your solution and explain your key decisions."
                )
            ],
            "phase": "CODING",
            "ready_for_coding": True,
            "discussion_turns": new_turns,
            "discussion_assessment": assessment,
            "unmet_criteria": [],
            "transition_reason": "discussion_assessment_validated",
        }

    if new_turns >= MAX_DISCUSSION_TURNS:
        return {
            "messages": [
                AIMessage(
                    content="We will move to coding now. Please implement your approach and explain your choices clearly."
                )
            ],
            "phase": "CODING",
            "ready_for_coding": True,
            "discussion_turns": new_turns,
            "discussion_assessment": assessment,
            "unmet_criteria": unmet,
            "transition_reason": "discussion_turn_limit_reached",
        }

    model_messages = [
        SystemMessage(content=final_prompt),
        *prior_messages,
    ]
    res = base_llm.invoke(model_messages)
    print(f"AI message: {res.content}")

    return {
        "messages": [res],
        "phase": "PROBLEM_DISCUSSION",
        "ready_for_coding": False,
        "discussion_turns": new_turns,
        "discussion_assessment": assessment,
        "unmet_criteria": unmet,
        "transition_reason": "discussion_in_progress",
    }

In [ ]:
def discussion_router(state: InterviewAgentState):
    if state.get("timer_expired"):
        return "internal_evaluation_node"

    if state.get("ready_for_coding") or state.get("phase") == "CODING":
        return "coding_node"

    if state.get("discussion_turns", 0) >= MAX_DISCUSSION_TURNS:
        return "coding_node"

    return "problem_discussion_node"

In [ ]:
def coding_phase_node(state: InterviewAgentState) -> dict:
    coding_phase_prompt = """
    CURRENT PHASE: CODING

    Your responsibilities:
    - Ask the candidate to walk through their implementation.
    - Identify unclear or potentially incorrect logic.
    - Suggest running test cases if appropriate.
    - Encourage debugging through reasoning.
    """

    problem_statement = state.get("problem_statement") or ""
    problem_context = _format_problem_context(state)
    user_code = state.get("user_code") or ""

    final_prompt = (
        f"{BASE_PROMPT}\\n{coding_phase_prompt}\\n"
        f"{problem_context}\\n\\n"
        f"Problem Statement:\\n{problem_statement}\\n\\n"
        f"Current user code:\\n{user_code}"
    )

    prior_messages = list(state.get("messages", []))
    current_turns = state.get("coding_turns", 0)
    if not prior_messages or not isinstance(prior_messages[-1], HumanMessage):
        return {
            "phase": "CODING",
            "coding_turns": current_turns,
            "transition_reason": "awaiting_user_message",
        }

    latest_user_content = prior_messages[-1].content
    latest_user_text = latest_user_content if isinstance(latest_user_content, str) else str(latest_user_content)
    assessment = _assess_coding_state(latest_user_text, state)
    validated, unmet = _validate_criteria(assessment)

    new_turns = current_turns + 1
    min_turn_gate = new_turns >= MIN_CODING_TURNS
    if validated and min_turn_gate:
        return {
            "messages": [
                AIMessage(
                    content="Thanks. We can now move to review. Please evaluate your solution for edge cases, trade-offs, and optimizations."
                )
            ],
            "phase": "REVIEW",
            "ready_for_review": True,
            "coding_turns": new_turns,
            "coding_assessment": assessment,
            "unmet_criteria": [],
            "transition_reason": "coding_assessment_validated",
        }

    if new_turns >= MAX_CODING_TURNS:
        return {
            "messages": [
                AIMessage(
                    content="Let's move to review now. Please summarize improvements, edge cases, and final complexity."
                )
            ],
            "phase": "REVIEW",
            "ready_for_review": True,
            "coding_turns": new_turns,
            "coding_assessment": assessment,
            "unmet_criteria": unmet,
            "transition_reason": "coding_turn_limit_reached",
        }

    model_messages = [
        SystemMessage(content=final_prompt),
        *prior_messages,
    ]
    res = base_llm.invoke(model_messages)
    print(f"AI message: {res.content}")

    return {
        "messages": [res],
        "phase": "CODING",
        "ready_for_review": False,
        "coding_turns": new_turns,
        "coding_assessment": assessment,
        "unmet_criteria": unmet,
        "transition_reason": "coding_in_progress",
    }

In [ ]:
def coding_router(state: InterviewAgentState):
    if state.get("timer_expired"):
        return "internal_evaluation_node"

    if state.get("ready_for_review") or state.get("phase") == "REVIEW":
        return "review_node"

    if state.get("coding_turns", 0) >= MAX_CODING_TURNS:
        return "review_node"

    return "coding_node"

In [ ]:
def review_phase_node(state: InterviewAgentState) -> dict:
    review_phase_prompt = """
    CURRENT PHASE: REVIEW

    Your responsibilities:
    - Ask the candidate to review their solution for correctness and edge cases.
    - Discuss possible optimizations and trade-offs.
    - Ask for final time and space complexity summary.
    """

    problem_statement = state.get("problem_statement") or ""
    problem_context = _format_problem_context(state)
    user_code = state.get("user_code") or ""

    final_prompt = (
        f"{BASE_PROMPT}\\n{review_phase_prompt}\\n"
        f"{problem_context}\\n\\n"
        f"Problem Statement:\\n{problem_statement}\\n\\n"
        f"Current user code:\\n{user_code}"
    )

    prior_messages = list(state.get("messages", []))
    current_turns = state.get("review_turns", 0)
    if not prior_messages or not isinstance(prior_messages[-1], HumanMessage):
        return {
            "phase": "REVIEW",
            "review_turns": current_turns,
            "transition_reason": "awaiting_user_message",
        }

    latest_user_content = prior_messages[-1].content
    latest_user_text = latest_user_content if isinstance(latest_user_content, str) else str(latest_user_content)
    assessment = _assess_review_state(latest_user_text, state)
    validated, unmet = _validate_criteria(assessment)

    new_turns = current_turns + 1
    min_turn_gate = new_turns >= MIN_REVIEW_TURNS
    if validated and min_turn_gate:
        return {
            "messages": [
                AIMessage(
                    content="Great, we have enough information. I will now generate final evaluation and feedback."
                )
            ],
            "phase": "FEEDBACK",
            "ready_for_feedback": True,
            "review_turns": new_turns,
            "review_assessment": assessment,
            "unmet_criteria": [],
            "transition_reason": "review_assessment_validated",
        }

    if new_turns >= MAX_REVIEW_TURNS:
        return {
            "messages": [
                AIMessage(
                    content="We are wrapping up now. I will generate your final evaluation and feedback."
                )
            ],
            "phase": "FEEDBACK",
            "ready_for_feedback": True,
            "review_turns": new_turns,
            "review_assessment": assessment,
            "unmet_criteria": unmet,
            "transition_reason": "review_turn_limit_reached",
        }

    model_messages = [
        SystemMessage(content=final_prompt),
        *prior_messages,
    ]
    res = base_llm.invoke(model_messages)
    print(f"AI message: {res.content}")

    return {
        "messages": [res],
        "phase": "REVIEW",
        "ready_for_feedback": False,
        "review_turns": new_turns,
        "review_assessment": assessment,
        "unmet_criteria": unmet,
        "transition_reason": "review_in_progress",
    }

In [ ]:
def review_router(state: InterviewAgentState):
    if state.get("timer_expired"):
        return "internal_evaluation_node"

    if state.get("ready_for_feedback") or state.get("phase") == "FEEDBACK":
        return "internal_evaluation_node"

    if state.get("review_turns", 0) >= MAX_REVIEW_TURNS:
        return "internal_evaluation_node"

    return "review_node"

In [ ]:
def _normalize_structured_output(result: Any, parsed_key: str = "parsed") -> Optional[Dict[str, Any]]:
    candidate = result.get(parsed_key) if isinstance(result, dict) and parsed_key in result else result
    if hasattr(candidate, "model_dump"):
        payload = candidate.model_dump()
        return payload if isinstance(payload, dict) else None
    if isinstance(candidate, dict):
        return candidate
    return None

def evaluation_node(state: InterviewAgentState) -> dict:
    problem_statement = state.get("problem_statement") or ""
    problem_references = state.get("problem_references") or {}
    user_code = state.get("user_code") or ""

    transcript_parts = []
    for message in state.get("messages", []):
        role = "unknown"
        if isinstance(message, HumanMessage):
            role = "candidate"
        elif isinstance(message, AIMessage):
            role = "interviewer"
        elif isinstance(message, SystemMessage):
            role = "system"

        content = getattr(message, "content", str(message))
        if isinstance(content, list):
            content = " ".join(str(item) for item in content)
        transcript_parts.append(f"[{role}] {content}")
    transcript = "\n".join(transcript_parts)

    eval_prompt = f"""
    CURRENT PHASE: INTERNAL_EVALUATION

    Generate an internal technical evaluation from the interview transcript.
    Focus only on technical reasoning, code quality, complexity understanding, and communication quality.


    Problem Statement:
    {problem_statement}

    Problem References:
    {problem_references}

    User Code:
    {user_code}

    Transcript:
    {transcript}
    """

    structured_llm = base_llm.with_structured_output(InternalEvaluationFormat)
    messages = [
        SystemMessage(content=eval_prompt),
        HumanMessage(content="Please perform the internal evaluation."),
    ]
    res = structured_llm.invoke(messages)
    internal_evaluation = _normalize_structured_output(res)
    if internal_evaluation is None:
        raise ValueError("Unexpected internal evaluation structured output type")

    return {
        "internal_evaluation": internal_evaluation,
        "phase": "FEEDBACK",
    }

In [ ]:
def feedback_phase_node(state: InterviewAgentState) -> dict:
    internal_evaluation = state.get("internal_evaluation") or {}

    problem_statement = state.get("problem_statement") or ""
    problem_references = state.get("problem_references") or {}
    user_code = state.get("user_code") or ""

    difficulty = state.get("difficulty")
    expected_time_minutes = int(state.get("expected_time_minutes") or 30)
    total_time_spent_sec = int(state.get("total_time_spent_sec") or 0)
    total_submissions = int(state.get("total_submissions") or 0)
    hints_used = int(state.get("hints_used") or 0)

    speed_percentile = max(
        0,
        min(
            100,
            100 - int((total_time_spent_sec / max(1, expected_time_minutes * 60)) * 100),
        ),
    )

    internal_evaluation_payload = (
        internal_evaluation.model_dump()
        if hasattr(internal_evaluation, "model_dump")
        else internal_evaluation
    )
    internal_evaluation_pretty = json.dumps(
        internal_evaluation_payload,
        indent=2,
        default=str,
    )

    feedback_prompt = f"""
    CURRENT PHASE: FEEDBACK

    Generate final interview feedback using the provided internal evaluation as source of truth.
    Ensure consistency across scores, strengths/weaknesses, metrics, and final verdict.


    Internal Evaluation:
    {internal_evaluation_pretty}

    Session metrics:
    - difficulty: {difficulty}
    - expected_time_minutes: {expected_time_minutes}
    - total_time_spent_sec: {total_time_spent_sec}
    - total_submissions: {total_submissions}
    - hints_used: {hints_used}
    - coding_speed_percentile: {speed_percentile}

    Problem Statement:
    {problem_statement}

    Problem References:
    {problem_references}

    User Code:
    {user_code}
    """

    structured_llm = base_llm.with_structured_output(
        FeedbackResponseFormat,
        include_raw=True,
    )
    messages = [
        SystemMessage(content=feedback_prompt),
        HumanMessage(content="Please generate the final interview feedback."),
    ]
    res = structured_llm.invoke(messages)

    payload = _normalize_structured_output(res)
    if payload is None:
        retry_prompt = (
            feedback_prompt
            + "\n\nIMPORTANT: Return output that strictly matches the FeedbackResponseFormat schema."
        )
        retry_messages = [
            SystemMessage(content=retry_prompt),
            HumanMessage(content="Retry and return strict structured output only."),
        ]
        retry_res = base_llm.with_structured_output(FeedbackResponseFormat).invoke(retry_messages)
        payload = _normalize_structured_output(retry_res)

    if payload is None:
        raw_content = None
        if isinstance(res, dict):
            raw = res.get("raw")
            raw_content = getattr(raw, "content", None)
        raise ValueError(f"Structured payload parse failed. Raw content: {raw_content}")

    response_text = payload.get("response")
    feedback_payload = payload.get("feedback")
    if not isinstance(response_text, str) or not response_text.strip():
        raise ValueError("Structured payload missing a valid response string")

    if hasattr(feedback_payload, "model_dump"):
        feedback_payload = feedback_payload.model_dump()
    if not isinstance(feedback_payload, dict):
        raise ValueError("Structured payload missing a valid feedback object")

    return {
        "feedback": feedback_payload,
        "messages": [AIMessage(content=response_text)],
        "phase": "FEEDBACK",
    }

In [ ]:
def create_interview_graph(checkpointer_obj: MemorySaver):
    workflow = StateGraph(InterviewAgentState)

    workflow.add_node("init_node", interview_init_node)
    workflow.add_node("problem_discussion_node", problem_discussion_node)
    workflow.add_node("coding_node", coding_phase_node)
    workflow.add_node("review_node", review_phase_node)
    workflow.add_node("internal_evaluation_node", evaluation_node)
    workflow.add_node("feedback_generation_node", feedback_phase_node)

    workflow.set_entry_point("init_node")
    workflow.add_edge("init_node", "problem_discussion_node")

    workflow.add_edge("internal_evaluation_node", "feedback_generation_node")
    workflow.add_edge("feedback_generation_node", END)

    workflow.add_conditional_edges(
        source="problem_discussion_node",
        path=discussion_router,
        path_map={
            "internal_evaluation_node": "internal_evaluation_node",
            "coding_node": "coding_node",
            "problem_discussion_node": "problem_discussion_node",
        },
    )

    workflow.add_conditional_edges(
        source="coding_node",
        path=coding_router,
        path_map={
            "internal_evaluation_node": "internal_evaluation_node",
            "review_node": "review_node",
            "coding_node": "coding_node",
        },
    )

    workflow.add_conditional_edges(
        source="review_node",
        path=review_router,
        path_map={
            "internal_evaluation_node": "internal_evaluation_node",
            "review_node": "review_node",
        },
    )

    return workflow.compile(
        checkpointer=checkpointer_obj,
        interrupt_after=["problem_discussion_node", "coding_node", "review_node"],
    )

In [ ]:
app = create_interview_graph(checkpointer)

In [ ]:
# Graph is built through create_interview_graph(checkpointer)
# This keeps notebook re-runs idempotent without internal API checks.

In [ ]:
# app is created in the previous cell.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
thread_id = f"session_{uuid4().hex[:8]}"
run_config = {"configurable": {"thread_id": thread_id}}

initial_state: InterviewAgentState = {
    "session_id": thread_id,
    "user_id": "user_mock_001",
    "phase": "PROBLEM_DISCUSSION",
    "messages": [],
    "problem_statement": "Given an integer array nums and an integer target, return indices of the two numbers such that they add up to target.",
    "problem_references": {
        "title": "Two Sum",
        "difficulty": "Easy",
        "optimal_approach": "Use a hash map to store seen values and their indices.",
        "time_complexity": "O(n)",
        "space_complexity": "O(n)",
    },
    "user_code": "def twoSum(nums, target):",
    "user_code_output": None,
    "execution_attempts": 0,
    "discussion_turns": 0,
    "review_turns": 0,
    "coding_turns": 0,
    "discussion_assessment": None,
    "coding_assessment": None,
    "review_assessment": None,
    "transition_reason": None,
    "unmet_criteria": [],
    "internal_evaluation": None,
    "feedback": None,
    "difficulty": "Easy",
    "expected_time_minutes": 20,
    "total_time_spent_sec": 0,
    "total_submissions": 0,
    "hints_used": 0,
    "interview_start_time": 0.0,
    "timer_expired": False,
    "ready_for_coding": False,
    "ready_for_review": False,
    "ready_for_feedback": False,
}

def get_thread_snapshot(config: dict) -> dict:
    snapshot = app.get_state(config)
    values = dict(snapshot.values) if getattr(snapshot, "values", None) else {}
    next_nodes = list(getattr(snapshot, "next", ()) or ())
    last_message = values.get("messages", [])[-1] if values.get("messages") else None
    last_message_type = type(last_message).__name__ if last_message is not None else None
    return {
        "thread_id": config.get("configurable", {}).get("thread_id"),
        "phase": values.get("phase"),
        "transition_reason": values.get("transition_reason"),
        "unmet_criteria": values.get("unmet_criteria"),
        "ready_for_coding": values.get("ready_for_coding"),
        "ready_for_review": values.get("ready_for_review"),
        "ready_for_feedback": values.get("ready_for_feedback"),
        "discussion_turns": values.get("discussion_turns"),
        "coding_turns": values.get("coding_turns"),
        "review_turns": values.get("review_turns"),
        "discussion_assessment": values.get("discussion_assessment"),
        "coding_assessment": values.get("coding_assessment"),
        "review_assessment": values.get("review_assessment"),
        "next_nodes": next_nodes,
        "last_message_type": last_message_type,
    }

def _resume_finalization_if_needed(config: dict, max_steps: int = 3) -> dict:
    for _ in range(max_steps):
        snapshot = app.get_state(config)
        values = dict(snapshot.values) if getattr(snapshot, "values", None) else {}
        next_nodes = list(getattr(snapshot, "next", ()) or ())

        if not next_nodes:
            return values

        should_resume = any(
            node in {"internal_evaluation_node", "feedback_generation_node"}
            for node in next_nodes
        )

        if not should_resume:
            return values

        app.invoke(Command(resume=True), config=config)

    raise RuntimeError("Finalization did not complete within max_steps")

def start_interview() -> dict:
    snapshot = app.get_state(run_config)
    values = dict(snapshot.values) if getattr(snapshot, "values", None) else {}
    if not values:
        app.invoke(initial_state, config=run_config)

    state = _resume_finalization_if_needed(run_config)
    return {
        "state": state,
        "debug": get_thread_snapshot(run_config),
    }

def continue_with_user(user_text: str, user_code: Optional[str] = None) -> dict:
    snapshot = app.get_state(run_config)
    current_values = dict(snapshot.values) if getattr(snapshot, "values", None) else {}
    current_phase = current_values.get("phase")
    state_update: Dict[str, Any] = {
        "messages": [HumanMessage(content=user_text)],
    }

    effective_user_code = user_code if user_code is not None else None
    if effective_user_code is None and _looks_like_code_submission(user_text):
        effective_user_code = user_text.strip()

    code_updated = False
    if effective_user_code is not None:
        state_update["user_code"] = effective_user_code
        code_updated = True
        if current_phase in {None, "PROBLEM_DISCUSSION"}:
            state_update["phase"] = "CODING"
            state_update["ready_for_coding"] = True
            state_update["transition_reason"] = "code_submission_detected"

    app.invoke(
        state_update,
        config=run_config,
    )
    state = _resume_finalization_if_needed(run_config)
    debug = get_thread_snapshot(run_config)
    debug["code_updated"] = code_updated
    return {
        "state": state,
        "debug": debug,
    }

turn_result = start_interview()

In [ ]:
continue_with_user(user_text="""
 i think my code is done now
                   
""")

In [ ]:
get_thread_snapshot(run_config)